## Feature Generation for Learning-to-Rank (LTR)
### Learn To Build LTR Feature Set (user features, item features, interaction label, and group_id)

This exercise is a continuation of the previous exercise. In previous notebooks, we created temporal train/val/test splits,  we implemented the evaluation harness, we built baseline recommenders and leaderboard and trained an Implicit MF (ALS) model and exported embeddings.

In this notebook, we build the **Learning-to-Rank (LTR) Feature Dataset**, which will power downstream ranking models (e.g., LightGBM, neural ranking models).

### Agenda

1. Load ALS user & item vectors  
2. Load train/valid/test splits  
3. Build LTR feature rows:
   - `user_id`
   - `item_id`
   - `label` (click / positive signal)
   - `group_id` (query grouping)  
   - dense user vector  
   - dense item vector  
4. Export `ltr_train.parquet`, `ltr_valid.parquet`, `ltr_test.parquet`

### Key Takeaways

By the end of this exercise, we will understand:
- How to convert implicit feedback interactions into supervised LTR samples  
- How to attach dense representation features to users & items  
- How to prepare query groups for pointwise or groupwise ranking models  
- How to build standardized feature files for downstream training

### Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

### Load ALS Embeddings

In [3]:
import json
from pathlib import Path

base_path = (
    "/content/drive/MyDrive/upgrad_live_sessions/"
    "Recommendation_systems/notebook-1/C6/data_splits/"
)

In [4]:
# base_path = "/content/"
user_vecs = np.load(base_path + "als_user_vectors.npy")     # shape (num_users, factors)
item_vecs = np.load(base_path + "als_item_vectors.npy")     # shape (num_items, factors)

user_vecs.shape, item_vecs.shape

((6040, 64), (3503, 64))

### Load Train / Valid / Test Splits

In [5]:
train = pd.read_csv(base_path +"train_20251103_1426.csv")
valid = pd.read_csv(base_path +"val_20251103_1426.csv")
test  = pd.read_csv(base_path +"test_20251103_1426.csv")

# standardize columns
for df in (train, valid, test):
    df.rename(columns={"UserID":"user_id", "MovieID":"item_id"}, inplace=True)

train.head()

,user_id,item_id,Rating,Timestamp,Datetime,label
0,1,3186,4,978300019,2000-12-31 22:00:19+00:00,True
1,1,1270,5,978300055,2000-12-31 22:00:55+00:00,True
2,1,1721,4,978300055,2000-12-31 22:00:55+00:00,True
3,1,1022,5,978300055,2000-12-31 22:00:55+00:00,True
4,1,2340,3,978300103,2000-12-31 22:01:43+00:00,False


### Build User / Item Mappings

These must match the mappings used in the previous notebook when building ALS. For safety, we rebuild them in first-appearance order from the train set.

In [6]:
user2idx = {}
item2idx = {}
idx2user = {}
idx2item = {}

uid_counter = 0
iid_counter = 0

for _, r in train.iterrows():
    u, i = r.user_id, r.item_id
    if u not in user2idx:
        user2idx[u] = uid_counter
        idx2user[uid_counter] = u
        uid_counter += 1
    if i not in item2idx:
        item2idx[i] = iid_counter
        idx2item[iid_counter] = i
        iid_counter += 1

len(user2idx), len(item2idx)

(6040, 3503)

### Utility Function: Attach User & Item Embeddings to Rows

In [7]:
def build_feature_row(df):
    rows = []

    for _, r in df.iterrows():
        u, i = r.user_id, r.item_id

        # skip if OOV (rare but safe)
        if u not in user2idx or i not in item2idx:
            continue

        uidx = user2idx[u]
        iidx = item2idx[i]

        uvec = user_vecs[uidx]
        ivec = item_vecs[iidx]

        row = {
            "user_id": u,
            "item_id": i,
            "label": int(r.label),       # already boolean TRUE/FALSE
            "group_id": u,               # group by user for LTR
        }

        # flatten vectors into individual feature columns
        for j, v in enumerate(uvec):
            row[f"user_vec_{j}"] = v
        for j, v in enumerate(ivec):
            row[f"item_vec_{j}"] = v

        rows.append(row)

    return pd.DataFrame(rows)

### Build LTR Datasets

In [8]:
ltr_train = build_feature_row(train)
ltr_valid = build_feature_row(valid)
ltr_test  = build_feature_row(test)

ltr_train.shape, ltr_valid.shape, ltr_test.shape

((987837, 132), (6040, 132), (6040, 132))

In [9]:
# checking the label distributions
display(ltr_train['label'].value_counts())
display(ltr_valid['label'].value_counts())
display(ltr_test['label'].value_counts())

,count
label,
1,568129
0,419708


,count
label,
1,3486
0,2554


,count
label,
1,3563
0,2477


### Save LTR Feature Files

We will save the LTR features as parquet files, as parquet files are compressed versions of the in-memory feature values.

In [10]:
ltr_train.to_parquet("ltr_train.parquet")
ltr_valid.to_parquet("ltr_valid.parquet")
ltr_test.to_parquet("ltr_test.parquet")

print("Saved ltr_train.parquet, ltr_valid.parquet, ltr_test.parquet")

Saved ltr_train.parquet, ltr_valid.parquet, ltr_test.parquet


In [11]:
import json
from pathlib import Path

base_path = Path(
    "/content/drive/MyDrive/upgrad_live_sessions/"
    "Recommendation_systems/notebook-1/C6/data_splits/"
)

base_path.mkdir(parents=True, exist_ok=True)

ltr_train.to_parquet(base_path / "ltr_train.parquet")
ltr_valid.to_parquet(base_path / "ltr_valid.parquet")
ltr_test.to_parquet(base_path / "ltr_test.parquet")

print("Saved ltr_train.parquet, ltr_valid.parquet, ltr_test.parquet to:", base_path)

Saved ltr_train.parquet, ltr_valid.parquet, ltr_test.parquet to: /content/drive/MyDrive/upgrad_live_sessions/Recommendation_systems/notebook-1/C6/data_splits


### Conclusion

In this exercise, we constructed the full Learning-to-Rank feature dataset by combining ALS user embeddings, ALS item embeddings, and implicit interaction labels derived from the temporal splits. Each user–item pair now has a **label** indicating positive interaction, a **group_id** allowing LTR models to treat each user as a ranked query, a **user embedding vector**, an **item embedding vector**. These feature files (`ltr_train.parquet`, `ltr_valid.parquet`, `ltr_test.parquet`) will be consumed by the next notebook, where we train and evaluate an LTR model such as LightGBM, or a neural ranker.